# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、严谨、带类比（analogy）的解释
- **额外要求**：云端路径用**流式（streaming）**一边生成一边更新 Markdown 显示

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | `system_question` 定角色，`user_question + question` 放具体问题 |
| 流式输出 `stream=True` | 逐 chunk 拼接，用 `update_display` 刷新 |
| OpenAI 云端模型 | `MODEL_GPT = 'gpt-4o-mini'` |
| 本地 Ollama | `ollama.chat(MODEL_LLAMA, ...)`（本格非流式，一次取完整回答） |

## 怎么跑

1. 配好 `.env`（至少能让 `OpenAI()` 读到密钥），并确保本机 Ollama 已安装 `llama3.2`
2. 按顺序跑导入 → 常量 → `load_dotenv` → 填写 `question`
3. 分别跑「路径 A：GPT 流式」和「路径 B：Llama」做对比


In [8]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI
# 从 os 只导入 getenv：按需读单个环境变量（本笔记本后面未必用到，但保持原导入）
from os import getenv
# 从 IPython.display 导入展示工具：display / Markdown / update_display（流式刷新用）
from IPython.display import display,Markdown,update_display
# 导入 ollama 官方 Python 包：直接调本地模型，不必自己拼 HTTP
import ollama


In [9]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'


In [10]:
# ========== 配置环境：读密钥并创建 OpenAI 客户端 ==========

# 加载 .env；override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 创建默认 OpenAI 客户端：密钥通常来自环境变量 OPENAI_API_KEY
openai = OpenAI()


In [11]:
# ========== 提问与提示词：改 question 就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再分别跑下面 GPT / Llama 两格做对比
question = """
Please explain what this code does and why: 
yield from {book.get("author") for book in books if book.get("author")}
"""

# system 角色：定「你是谁、怎么答」——专家口吻 + 要求详细回答（字符串保持英文原样）
system_question = """You are expert at understanding data science, machine learning, python and coding of any kind. 
Give me a detailed response."""

# user 侧前缀：要求详细回答 + 类比（analogy）+ 事实可核对；后面会拼上真正的 question
user_question = """
    For the question provided by the user give me a detailed response along with a anology
    in layman terms. Ensure that the response provided is factual and is referenced from a valid
    source of truth. Here is the question provided by the user:
    """


In [12]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# 封装一次完整的「带 system/user 的流式问答」，边收边用 Markdown 刷新显示
def answer_with_openai(question,system_question,user_question):
    # 调用 Chat Completions；stream=True 返回可迭代的 chunk，而不是整段一次性结果
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            # system：角色与回答风格
            {"role":"system","content":system_question},
            # user：把说明前缀和具体代码问题拼在一起
            {"role":"user","content":user_question + question}
        ],
        stream=True
        )
    # 累积已经收到的文本，供 Markdown 整段重绘
    response = ""
    # 先放一个空的 Markdown 占位，拿到 display_id，后面才能原地更新同一块区域
    my_display = display(Markdown(""),display_id=True)
    # 逐块遍历流式结果
    for chunk in stream:
        # delta.content 可能为 None（例如结束标记），有字才追加
        if chunk.choices[0].delta.content is not None:
            response += chunk.choices[0].delta.content
            # 用同一 display_id 刷新：视觉上像「打字机」不断变长
            update_display(Markdown(response),display_id=my_display.display_id)

# 真正发起提问：沿用上一格定义的 question / system_question / user_question
answer_with_openai(question,system_question,user_question)


Certainly! Let's break down the provided code snippet and explain what it does step by step, along with an analogy to simplify the concept.

### Code Explanation

The given line of code is:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

1. **Set Comprehension**: 
   - The part inside the curly braces, `{book.get("author") for book in books if book.get("author")}`, is a *set comprehension*. It creates a set of unique authors from a list (or iterable) called `books`.
   - The expression `book.get("author")` attempts to retrieve the author of each `book` dictionary. The `get` method is used because it safely returns `None` if the "author" key does not exist instead of raising an error.
   - The `if book.get("author")` condition filters out any books that do not have an author, ensuring that only valid author names are included in the set.

2. **Yielding Values**: 
   - The `yield from` statement is used to yield all items from an iterable (in this case, the set of authors) to the caller of a generator function. When a function uses `yield`, it becomes a generator function. Instead of returning a single value, it can yield multiple values over time.
   - Essentially, `yield from` allows you to yield each author one by one, rather than returning the entire set at once.

### What It Does

Overall, this line of code extracts all unique authors from the `books` collection where the author information is present and yields each author's name. This could be part of a generator function that, when iterated over, would give you each author one by one.

### Analogy in Layman Terms

Imagine you are at a library (the `books` collection), and you are tasked with making a list of all the authors whose books are available. 

1. **Identifying Books**: You pick up each book one by one and check if there is an author listed inside (that's like using `book.get("author")`).
   
2. **Making a Unique List**: If a book has an author listed, you add that author's name to a special list. However, to keep things neat, if you find out you already added that author before, you won’t add them again (that’s what using a set does—it prevents duplicates).

3. **Sharing the List**: Once you’ve gone through all the books, instead of handing the entire list to someone at once, you decide to read each author's name out loud one at a time (this is akin to using `yield from`).

### Source of Truth

For further details about Python's **set comprehension** and **generators**, you can refer to the Python official documentation:

- **Set Comprehensions**: [Python Documentation on Set Comprehensions](https://docs.python.org/3/tutorial/datastructures.html#set-comprehensions)
- **Generators and Yield**: [Python Documentation on Defining Functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions)

In summary, this code is an efficient way to gather unique authors from a list of books and to yield each author one by one as needed, thereby promoting memory efficiency and ease of use in iterating through large datasets.

In [ ]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）一次取完整回答 ==========

# 调用 ollama.chat：第一个参数是模型名；messages 结构与 OpenAI 风格一致
# 理念：同一问题、两个后端 —— 对比云端流式 API 与本地开源模型的速度、风格、是否需要密钥
# 前提：Ollama 在跑，且已安装 llama3.2
response = ollama.chat(
    MODEL_LLAMA,
    messages=[
            {"role":"system","content":system_question},
            {"role":"user","content":user_question + question}
        ]   
    )

# Ollama 返回字典；回答正文在 message.content
ollama_response = response["message"]["content"]
# 在笔记本里用 Markdown 渲染完整回答（本路径非流式）
display(Markdown(ollama_response))
